# Embeddings

Has aprendido a procesar texto tokenizándolo y convirtiéndolo en tensores, un paso necesario para cualquier red neuronal. Sin embargo, los IDs enteros asignados a las palabras no capturan inherentemente su significado o sus relaciones. El número para "cat" (gato) no está más relacionado con el número para "dog" (perro) de lo que está con el de "banana". Aquí es donde entran en juego los **word embeddings** (incrustaciones de palabras). Los embeddings son representaciones vectoriales densas que mapean palabras en un espacio multidimensional donde las relaciones semánticas pueden medirse matemáticamente.

En este laboratorio, obtendrás experiencia práctica con los conceptos centrales detrás de los embeddings y verás cómo se implementan en PyTorch.

* Comenzarás cargando **embeddings GloVe preentrenados** para explorar cómo capturan la similitud semántica y resuelven analogías como `king - man + woman ≈ queen` (rey - hombre + mujer ≈ reina).
* A continuación, **visualizarás** estos vectores de alta dimensión en un espacio 2D para ver cómo los conceptos relacionados forman grupos (clusters) distintos.



* Luego, **construirás un modelo de embedding simple desde cero**, definiendo tu propio vocabulario y entrenándolo para aprender relaciones de palabras en una tarea específica.
* Finalmente, investigarás las limitaciones de los embeddings estáticos con palabras que tienen múltiples significados y verás cómo los **modelos contextuales como BERT** proporcionan una solución más dinámica y potente.

## Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from itertools import permutations

import helper_utils

# Set random seeds for reproducibility
torch.manual_seed(123)
np.random.seed(123)

## Word Embeddings (Incrustaciones de palabras)

Como ya sabes, en su esencia, los modelos de aprendizaje automático entienden números, no palabras. Los **Word Embeddings** son la solución a este problema para los modelos basados en texto. Son representaciones vectoriales numéricas de palabras que están diseñadas para capturar su significado semántico, sus relaciones y su contexto.



Existen dos categorías principales de word embeddings:

* **Static Embeddings (Embeddings estáticos)**: Este es el enfoque clásico donde cada palabra del vocabulario se mapea a un único vector fijo. Modelos como **Word2Vec** y **GloVe** utilizan este método. El vector para la palabra "Apple" es el mismo, independientemente de si estás hablando de la fruta o de la compañía tecnológica.

* **Dynamic (or Contextual) Embeddings (Embeddings dinámicos o contextuales)**: Este es un enfoque más avanzado donde el vector de una palabra cambia según la oración en la que se encuentre. Modelos como **BERT** sobresalen en esto, generando diferentes vectores para "Apple" dependiendo del contexto que la rodea.

[Image comparing static embeddings where a word has one vector versus contextual embeddings where a word has multiple vectors based on context]

Comenzarás analizando un potente modelo de embedding estático preentrenado llamado **GloVe**.

### GloVe: Static Embeddings

[GloVe](https://nlp.stanford.edu/projects/glove/), que significa **Global Vectors for Word Representation** (Vectores Globales para Representación de Palabras), es uno de los modelos de embedding estático preentrenados más populares. Es un algoritmo de aprendizaje no supervisado que aprende vectores de palabras analizando un corpus masivo de texto y calculando estadísticas agregadas de co-ocurrencia global palabra-palabra, aprendiendo esencialmente de qué tan frecuentemente aparecen las palabras cerca unas de otras.

Los creadores de GloVe proporcionan varios modelos preentrenados, cada uno entrenado en diferentes conjuntos de datos a gran escala:

* **glove.6B**: Entrenado en Wikipedia y texto de Gigaword, contiene 6 mil millones de tokens y un vocabulario de 400,000 palabras.

* **glove.42B** y **glove.840B**: Entrenados en un conjunto masivo de datos web (Common Crawl) con 42 mil millones y 840 mil millones de tokens, respectivamente. Estos tienen vocabularios mucho más grandes (1.9M y 2.2M de palabras).

* **glove.twitter.27B**: Un modelo entrenado específicamente en 2 mil millones de tweets con 27 mil millones de tokens.

Si bien los modelos más grandes entrenados en Common Crawl ofrecen vocabularios vastos, el modelo **glove.6B** es un punto de partida extremadamente popular para muchas tareas de propósito general debido a su tamaño manejable y su potente rendimiento.

Utilizarás esta versión **glove.6B**. Ejecuta las siguientes celdas para descargar y cargar los embeddings de GloVe preentrenados.

In [ ]:
# Descargar los datos para el modelo GloVe 6B 100d
helper_utils.download_glove6B()

In [ ]:
# Especificar la ruta al archivo GloVe de 100 dimensiones (100d)
glove_file = './glove_data/glove.6B.100d.txt'

# Cargar los vectores de palabras preentrenados desde el archivo
glove_embeddings = helper_utils.load_glove_embeddings(glove_file)

#### Similitud semántica con GloVe

Una de las características más potentes de los *word embeddings* es su capacidad para capturar la **similitud semántica**. Esto significa que las palabras con significados similares tendrán representaciones vectoriales que están cerca unas de otras en el espacio vectorial.

Puedes medir esta cercanía utilizando una métrica llamada **similitud de coseno** (*cosine similarity*). Esta métrica calcula el coseno del ángulo entre dos vectores. Una similitud de coseno más alta (cercana a 1) indica que las palabras están más relacionadas semánticamente, mientras que un valor más bajo (cercano a 0 o -1) sugiere que no lo están. Por ejemplo, se esperaría que el vector de "cat" (gato) tenga una alta similitud de coseno con "dog" (perro), pero una similitud muy baja con "car" (auto).

Más allá de la simple similitud, estos embeddings también pueden capturar relaciones y analogías más complejas. El ejemplo más famoso es que la relación vectorial entre `king` (rey) y `man` (hombre) es similar a la que existe entre `queen` (reina) y `woman` (mujer). Puedes probar esto con aritmética de vectores: `king - man + woman ≈ queen`.

* Define la función `find_closest_words`, que toma un vector objetivo, el diccionario completo de embeddings, una lista opcional de palabras para excluir y un parámetro `top_n`. Su objetivo es devolver una lista de las `n` palabras más similares semánticamente junto con sus puntuaciones de similitud.
* Para encontrar las coincidencias semánticas más cercanas, esta función utiliza un enfoque vectorizado altamente eficiente para calcular la similitud de coseno entre el vector objetivo y todos los demás vectores de palabras en una sola operación. Ordena las puntuaciones de similitud para encontrar las mejores `n` coincidencias y las devuelve con sus palabras correspondientes.

In [ ]:
def find_closest_words(embedding, embeddings_dict, exclude_words=[], top_n=5):
    """
    Encuentra las N palabras más semánticamente similares a un vector dado y sus puntuaciones.

    Argumentos:
        embedding: La representación vectorial de la palabra objetivo.
        embeddings_dict: Un diccionario que mapea palabras a sus vectores de embedding.
        exclude_words: Una lista de palabras a excluir de la búsqueda.
        top_n: El número de palabras más similares a devolver.

    Devuelve:
        Una lista de tuplas, donde cada tupla contiene una palabra y su
        puntuación de similitud de coseno, ordenada de forma descendente por similitud.
        Devuelve None si el vocabulario está vacío después de las exclusiones.
    """
    # Filtrar el vocabulario para eliminar cualquier palabra en la lista de exclusión.
    filtered_words = [word for word in embeddings_dict.keys() if word not in exclude_words]
    
    # Manejar el caso extremo donde el vocabulario filtrado está vacío.
    if not filtered_words:
        return None
        
    # Crear una matriz de todos los vectores de palabras para un cálculo eficiente.
    embedding_matrix = np.array([embeddings_dict[word] for word in filtered_words])
    
    # Redimensionar el embedding objetivo a un arreglo 2D para la función de similitud.
    target_embedding = embedding.reshape(1, -1)
    
    # Calcular la similitud de coseno entre el objetivo y todas las demás palabras.
    similarity_scores = cosine_similarity(target_embedding, embedding_matrix)
    
    # Obtener los índices de las top N palabras con las puntuaciones de similitud más altas.
    closest_word_indices = np.argsort(similarity_scores[0])[::-1][:top_n]
    
    # Crear una lista de tuplas (palabra, puntuación) para las top N palabras más cercanas.
    return [(filtered_words[i], similarity_scores[0][i]) for i in closest_word_indices]

* Primero, verifica si las palabras "king", "man" y "woman" existen en el vocabulario de GloVe y recupera sus vectores correspondientes.

In [ ]:
# Asegurarse de que las palabras existan en glove_embeddings
if all(word in glove_embeddings for word in ['king', 'man', 'woman']):
    king = glove_embeddings['king']
    man = glove_embeddings['man']
    woman = glove_embeddings['woman']

* Ahora, realiza el cálculo vectorial `king - man + woman` para encontrar el embedding resultante para la analogía.

In [ ]:
# El vector resultante para la analogía
result_embedding = king - man + woman

* Finalmente, utiliza la función `find_closest_words` para buscar en el vocabulario la lista de palabras que son semánticamente más cercanas al `result_embedding` calculado.
    * Las palabras originales (`king`, `man`, `woman`) se excluyen de la búsqueda. Esto evita que la función simplemente devuelva una de las palabras de entrada, lo cual no sería una respuesta significativa a la analogía.

In [ ]:
# Establecer las N palabras principales
top_n = 5

# Encontrar las top N palabras más cercanas, asegurándose de excluir las entradas
closest_words_with_scores = find_closest_words(
    result_embedding, 
    glove_embeddings, 
    exclude_words=['king', 'man', 'woman'],
    top_n=top_n
)

# Verificar si se devolvieron palabras
if closest_words_with_scores:
    # Desempaquetar el mejor resultado (palabra y puntuación)
    top_word, top_score = closest_words_with_scores[0]

    # Imprimir el mejor resultado en el formato original con su puntuación
    print(f"king - man + woman ≈ {top_word} (Score: {top_score:.4f})")

    # Imprimir los otros 4 resultados
    if len(closest_words_with_scores) > 1:
        print(f"\n--- Otros {top_n-1} resultados principales ---")
        # Recorrer el resto de la lista de tuplas
        for word, score in closest_words_with_scores[1:]:
            print(f"{word} (Score: {score:.4f})")

<br>

Este resultado es una demostración poderosa de cómo los *word embeddings* capturan relaciones lingüísticas profundas, permitiéndote resolver analogías complejas con aritmética vectorial simple. Para ver más a fondo cómo estos vectores agrupan conceptos relacionados, ahora procederás a visualizarlos.

Siéntete libre de experimentar con otras analogías para ver qué otras relaciones ha aprendido el modelo. Por ejemplo, podrías intentar:

* `france - paris + tokyo` (para encontrar el país de una capital dada)

* `walking - walk + swim` (para encontrar el tiempo progresivo de un verbo)

* `uncle - man + woman` (para encontrar el equivalente femenino)

In [ ]:
# Define tu analogía

# Tu analogía seguirá este formato:
# word1 - word2 + word3 = ???

# Define tus palabras
word1 = ''
word2 = ''
word3 = ''

# Establece cuántos resultados principales quieres ver
top_n = 5

In [ ]:
# Una lista de las palabras utilizadas en la analogía
analogy_words = [word1, word2, word3]

# Verificar si todas las palabras existen en el diccionario de embeddings
if all(word in glove_embeddings for word in analogy_words):
    # Obtener el vector de embedding para cada palabra
    embedding1 = glove_embeddings[word1]
    embedding2 = glove_embeddings[word2]
    embedding3 = glove_embeddings[word3]

    # Realizar la aritmética vectorial para encontrar el embedding resultante
    result_embedding = embedding1 - embedding2 + embedding3

    # Encontrar las top N palabras más cercanas al resultado, excluyendo las palabras de entrada
    closest_words_with_scores = find_closest_words(
        result_embedding, 
        glove_embeddings, 
        exclude_words=analogy_words,
        top_n=top_n
    )

    # Verificar si la función devolvió alguna palabra similar
    if closest_words_with_scores:
        # Obtener la palabra principal y su puntuación de similitud
        top_word, top_score = closest_words_with_scores[0]

        # Imprimir la analogía y su mejor resultado
        print(f"'{word1}' - '{word2}' + '{word3}' ≈ '{top_word}' (Puntuación: {top_score:.4f})")

        # Verificar si hay otros resultados para mostrar
        if len(closest_words_with_scores) > 1:
            print(f"\n--- Otros {len(closest_words_with_scores) - 1} resultados principales ---")
            # Recorrer el resto de los resultados e imprimirlos
            for word, score in closest_words_with_scores[1:]:
                print(f"{word} (Puntuación: {score:.4f})")
    else:
        # Mensaje si no se encontraron palabras similares
        print("No se pudieron encontrar palabras similares para la analogía dada.")

else:
    # Buscar e informar qué palabras faltan en el vocabulario
    missing_words = [word for word in analogy_words if word not in glove_embeddings]
    print(f"Error: Las siguientes palabras no se encontraron en el vocabulario: {missing_words}")
    print("Por favor, intenta con palabras diferentes.")

#### Visualización de embeddings de GloVe

Hasta ahora, has confirmado estas relaciones semánticas numéricamente con aritmética vectorial. Otra forma poderosa de entender la estructura de estos embeddings es visualizarlos. Sin embargo, graficar un vector no es una tarea directa.

Cada word embedding es un vector con un número específico de **dimensiones**. Para el modelo GloVe que has cargado, son 100 dimensiones. Cada una de estas dimensiones captura una característica abstracta diferente del significado de la palabra y sus relaciones con otras. Aunque tener muchas dimensiones permite al modelo almacenar información rica y matizada, también crea un desafío: es imposible para nosotros visualizar directamente un espacio de 100 dimensiones.

Para resolver esto, puedes usar una técnica de reducción de dimensionalidad llamada **Análisis de Componentes Principales (PCA)**. El PCA analiza los datos para encontrar las direcciones de máxima varianza y proyecta los vectores originales de alta dimensión en un nuevo espacio de menor dimensión. En términos simples, te ayuda a "comprimir" las 100 dimensiones a solo dos (una coordenada x e y), preservando la mayor parte posible de la estructura semántica original.

Al hacer esto, puedes crear un gráfico en 2D para ver cómo GloVe agrupa conceptos relacionados.

* Para ver esto en acción, define una lista `words_to_visualize` que contenga las palabras que visualizarás.
    * Nota que las palabras en esta lista pueden agruparse en categorías claras: tipos de vehículos, tipos de mascotas y tipos de frutas.

In [ ]:
# Definir el vocabulario de palabras de diferentes categorías para visualizar.
words_to_visualize = ['car', 'bike', 'plane',      # Categoría: Vehículos
                      'cat', 'dog', 'bird',        # Categoría: Mascotas
                      'orange', 'apple', 'grape'   # Categoría: Frutas
]

# Un diccionario que agrupa las mismas palabras de `words_to_visualize` por categoría para facilitar la visualización
visualization_dict = {
    'Vehicle': ['car', 'bike', 'plane'],
    'Pet': ['cat', 'dog', 'bird'],
    'Fruit': ['orange', 'apple', 'grape']
}

* Ahora, recupera el vector de embedding de GloVe para cada palabra en la lista `words_to_visualize`.


In [ ]:
# Inicializar una lista vacía para almacenar los vectores.
embedding_vectors_list = []

# Recorrer cada palabra en la lista `words_to_visualize`.
for word in words_to_visualize:
    # Obtener el embedding de la palabra y añadirlo a la lista.
    embedding_vectors_list.append(glove_embeddings[word])

# Convertir la lista de vectores en un arreglo de NumPy.
embedding_vectors = np.array(embedding_vectors_list)

* Utiliza PCA para reducir los vectores de embedding de 100 dimensiones a solo dos dimensiones para que puedan ser graficados.
    * El parámetro `n_components=2` especifica que deseas reducir los datos a una representación bidimensional, creando una coordenada x e y para cada palabra.

In [ ]:
# Inicializar el modelo PCA para reducir las dimensiones a 2
reducer = PCA(n_components=2)

# Aplicar PCA a los vectores de embedding para obtener coordenadas 2D
coords_2d = reducer.fit_transform(embedding_vectors)

* Grafica los embeddings reducidos en 2D para ver cómo se agrupan las palabras.

    * **Nota**: Si has realizado algún cambio en la lista `words_to_visualize`, asegúrate de que se reflejen adecuadamente también en el diccionario `visualization_dict`; de lo contrario, el gráfico devolverá un error.

In [ ]:
helper_utils.plot_embeddings(coords=coords_2d, 
                             labels=words_to_visualize,
                             label_dict=visualization_dict,
                             title='GloVe Pre-Trained Embeddings'
                            )

El gráfico anterior confirma visualmente que el modelo GloVe comprende las relaciones entre palabras. Incluso después de comprimir los vectores de 100 dimensiones a solo dos, las palabras forman grupos (clusters) distintos basados en su categoría semántica:

* Los **vehículos** (`car`, `bike`, `plane`) están agrupados.
* Las **mascotas** (`cat`, `dog`, `bird`) forman un segundo grupo distinto.
* Las **frutas** (`orange`, `apple`, `grape`) crean un tercer grupo.

## Construyendo tus propios embeddings desde cero

Si bien los modelos preentrenados como GloVe son potentes, hay muchos casos en los que necesitas entrenar tus propios embeddings. Esto es especialmente cierto cuando trabajas con un vocabulario especializado (por ejemplo, términos médicos o financieros) que puede no estar presente en los modelos preentrenados, o cuando deseas capturar relaciones semánticas específicas de tu conjunto de datos.

En esta sección, aprenderás el proceso fundamental para entrenar embeddings. Comenzarás con un vocabulario pequeño y personalizado, y utilizarás PyTorch para construir un modelo simple que aprenda representaciones vectoriales desde cero.

### Definiendo el vocabulario y los parámetros

Antes de crear el modelo, debes establecer las piezas fundamentales. Esto implica definir el vocabulario específico de palabras que el modelo aprenderá y configurar los parámetros clave para la capa de embedding, como el tamaño del vocabulario y la dimensión de los vectores de embedding.

* Define la lista `vocabulary`. Esta contendrá todas las palabras de diferentes categorías semánticas que tu modelo aprenderá.

In [ ]:
vocabulary = ['car', 'bike', 'plane', 
              'cat', 'dog', 'bird', 
              'orange', 'apple', 'grape']

* Crea dos mapeos esenciales a partir de tu lista `vocabulary`: un diccionario `word_to_idx` para convertir palabras en índices numéricos, y un diccionario `idx_to_word` para convertir esos índices de nuevo en palabras.
    * Esto es similar a la función `build_vocab` del laboratorio anterior, pero es más directo porque estás comenzando con una lista predefinida de palabras únicas. Nota que esta vez la indexación comienza desde **0**.

In [ ]:
# Crear el mapeo de palabra a índice
# Inicializar un diccionario vacío para el mapeo word-to-index
word_to_idx = {}

# Recorrer la lista del vocabulario con un índice
for i, word in enumerate(vocabulary):
    # Asignar cada palabra a su índice correspondiente
    word_to_idx[word] = i
    
# Crear el mapeo de índice a palabra
# Inicializar un diccionario vacío para el mapeo index-to-word
idx_to_word = {}

# Recorrer los elementos del diccionario word_to_idx recién creado
for word, i in word_to_idx.items():
    # Asignar cada índice a su palabra correspondiente
    idx_to_word[i] = word

* Define `vocab_size` y `embedding_dim`, que son parámetros esenciales para el modelo.
    * `embedding_dim = 3`: Ten en cuenta que, aunque usamos una dimensión de embedding de 3 para este ejemplo simple, un rango típico para vocabularios más grandes es de 100 a 300.

In [ ]:
# Obtener el número total de palabras únicas en tu vocabulario
vocab_size = len(vocabulary)

# Definir el tamaño del vector de embedding para cada palabra
embedding_dim = 3

In [ ]:
# Imprimir el mapeo de palabra a índice para revisarlo
print("Vocabulario:\tÍndice:")
for word, idx in word_to_idx.items():
    print(f"{word}:\t\t{idx}")

# Imprimir los parámetros finales que se utilizarán para el modelo
print(f"\nTamaño del vocabulario: {vocab_size}")
print(f"Dimensión del embedding: {embedding_dim}")

### Generación de pares de entrenamiento

Ahora que has definido tu vocabulario, el siguiente paso es crear los datos de entrenamiento que enseñarán al modelo las relaciones entre estas palabras. El objetivo es que el modelo aprenda que las palabras dentro de la misma categoría son similares.

Puedes lograr esto con una tarea de predicción simple. El modelo aprende al recibir una palabra de entrada e intentar prevenir otra palabra de esa misma categoría. Esto requiere estructurar tu vocabulario en pares `(input_word, target_word)` que sirvan como ejemplos de entrenamiento.

* Primero, estructura tu vocabulario en categorías distintas utilizando un diccionario de Python. Este diccionario mapeará el nombre de una categoría (como 'Vehicles') a una lista de palabras pertenecientes a esa categoría.

In [ ]:
# Definir el vocabulario, agrupado por categoría semántica.
vocab_categories = {
    'Vehicles': ['car', 'bike', 'plane'],
    'Pets': ['cat', 'dog', 'bird'],
    'Fruits': ['orange', 'apple', 'grape']
}

* Para enseñarle al modelo que palabras como 'car', 'bike' y 'plane' están relacionadas, necesitas crear pares de entrenamiento como `('car', 'bike')`, `('car', 'plane')`, `('bike', 'plane')`, y así sucesivamente para cada categoría.
* En lugar de escribir cada combinación a mano, puedes usar la función `itertools.permutations` de Python para generar estos pares automáticamente.

In [ ]:
# Inicializar una lista vacía para contener los pares de entrenamiento.
training_pairs = []

# Iterar a través de las listas de palabras en el diccionario vocab_categories.
for category_list in vocab_categories.values():
    # Generar todas las permutaciones de 2 palabras de la lista y añadirlas a training_pairs.
    training_pairs.extend(list(permutations(category_list, 2)))

# Mostrar el número total de pares y una muestra de los pares generados.
print(f"Se generaron {len(training_pairs)} pares de entrenamiento.")
print("Pares generados:\n")
for pair in training_pairs:
    print(pair)

### Modelo de Embedding

Ahora que tienes tus datos de entrenamiento, necesitas un modelo que pueda aprender de ellos. Crearás una red neuronal sencilla con dos capas clave:

* `nn.Embedding`: Actúa como una tabla de búsqueda (lookup table) donde el índice de cada palabra (como el `3` para 'cat') se mapea a un vector específico. Durante el entrenamiento, el modelo utiliza este vector para realizar una predicción y luego lo ajusta para capturar mejor las relaciones de la palabra, convirtiendo el vector inicialmente aleatorio en uno significativo.

* **nn.Linear**: Esta es una capa estándar completamente conectada (fully connected). Tomará el vector de la palabra de la capa de embedding y producirá una predicción para la palabra objetivo.

* Define un `SimpleEmbeddingModel`.
    * El modelo se inicializa con `vocab_size` (tamaño del vocabulario) y `embedding_dim` (dimensión del embedding).
    * Contiene una **capa** `embedding` que funciona como la tabla de búsqueda de vectores.
    * También contiene una **capa** `linear` que genera las puntuaciones de predicción finales.
* Define el paso hacia adelante (`forward`) para el modelo.
    * Toma el índice de una palabra (`x`) como entrada.
    * Recupera el vector de la palabra desde la capa de `embedding`.
    * Pasa ese vector a la capa `linear` para obtener el resultado.

In [ ]:
class SimpleEmbeddingModel(nn.Module):
    """Un modelo de red neuronal simple para aprender embeddings de palabras."""
    def __init__(self, vocab_size, embedding_dim):
        """
        Inicializa las capas del modelo.

        Argumentos:
            vocab_size: El número total de palabras únicas en el vocabulario.
            embedding_dim: La dimensionalidad deseada de los embeddings de palabras.
        """
        # Llamar al constructor de la clase padre (nn.Module).
        super().__init__()
        
        # Una capa de embedding que mapea índices de palabras a vectores densos.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Una capa lineal que proyecta el vector de embedding al tamaño del vocabulario.
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        """
        Define el paso hacia adelante (forward pass) del modelo.

        Argumentos:
            x: Un tensor de índices de palabras de entrada.

        Devuelve:
            Una tupla que contiene los logits de salida de la capa lineal y
            los vectores de embedding intermedios.
        """
        # La entrada 'x' se pasa a través de la capa de embedding.
        embedded = self.embedding(x)
        # El vector de embedding resultante se pasa a través de la capa lineal.
        output = self.linear(embedded)
        
        return output, embedded

* Initialize the model.

In [ ]:
embedding_model = SimpleEmbeddingModel(vocab_size, embedding_dim)

### Optimizador y Función de Pérdida

* Define el optimizador `Adam`.
* Define la Entropía Cruzada (`CrossEntropyLoss`) como la función de pérdida.

In [ ]:
# Initialize the Adam optimizer
optimizer = torch.optim.Adam(embedding_model.parameters(), lr=0.01)

# Initialize the CrossEntropyLoss function
loss_function = nn.CrossEntropyLoss()

### Entrenando Embeddings con un Modelo Simple

Este bucle de entrenamiento sigue la misma lógica central que has visto antes: itera durante múltiples **épocas**, realiza un **paso hacia adelante (forward pass)**, calcula la **pérdida** y utiliza un **paso hacia atrás (backward pass)** para actualizar el modelo.

La diferencia clave radica en cómo maneja los datos. En lugar de trabajar con imágenes y etiquetas, este bucle está diseñado para pares de palabras.

* **Iteración de Datos**: En lugar de usar un `DataLoader` que entrega lotes de `(imagen, etiqueta)`, este bucle itera directamente sobre tu lista simple de pares `(word1, word2)`.

* **Conversión de Texto a Índice**: La diferencia más significativa es el paso de preparación de datos dentro del bucle. Para cada par, las palabras en cadena (ej. 'car', 'bike') deben convertirse en sus índices numéricos utilizando el mapeo `word_to_idx`. Este es el paso esencial para procesar texto.

* **Entrada y Objetivo**: En esta configuración, el índice de la primera palabra del par (`word1_idx`) es la entrada para el modelo. El índice de la segunda palabra (`word2_idx`) actúa como la etiqueta objetivo que la función de pérdida utiliza para medir el error.

In [ ]:
def training_loop(model, training_pairs, epochs=2000):
    """
    Entrena un modelo simple de word embeddings.

    Argumentos:
        model: El modelo de PyTorch a entrenar.
        training_pairs: Una lista de tuplas, donde cada tupla es un
                        par (palabra_entrada, palabra_objetivo).
        epochs: El número total de iteraciones de entrenamiento sobre el conjunto de datos.

    Devuelve:
        Una tupla que contiene el modelo entrenado y una lista de la
        pérdida promedio por cada época.
    """
    # Establecer el modelo en modo de entrenamiento.
    model.train()
    # Inicializar una lista para almacenar el valor de la pérdida de cada época.
    losses = []

    # Iterar sobre el conjunto de datos por un número especificado de épocas.
    for epoch in range(epochs):
        # Inicializar la pérdida total para la época actual.
        epoch_loss = 0

        # Iterar por cada par entrada-objetivo en los datos de entrenamiento.
        for word1, word2 in training_pairs:
            # Convertir las palabras (strings) en sus índices numéricos correspondientes.
            word1_idx = torch.tensor([word_to_idx[word1]])
            word2_idx = torch.tensor([word_to_idx[word2]])

            # Realizar un paso hacia adelante (forward pass) para obtener las predicciones del modelo.
            output, _ = model(word1_idx)
            # Calcular la pérdida entre las predicciones y el objetivo real.
            loss = loss_function(output, word2_idx)

            # Limpiar cualquier gradiente calculado previamente antes del paso hacia atrás.
            optimizer.zero_grad()
            # Calcular el gradiente de la pérdida respecto a los parámetros del modelo.
            loss.backward()
            # Actualizar los pesos del modelo basándose en los gradientes computados.
            optimizer.step()

            # Acumular la pérdida para la época actual.
            epoch_loss += loss.item()

        # Calcular la pérdida promedio de la época y almacenarla.
        losses.append(epoch_loss / len(training_pairs))

        # Imprimir periódicamente el progreso del entrenamiento.
        if epoch % 200 == 0:
            print(f"Época {epoch}, Pérdida: {losses[-1]:.4f}")

    # Imprimir la pérdida final una vez completado el entrenamiento.
    print(f"Época {epochs}, Pérdida: {losses[-1]:.4f}")

    return model, losses

* Ejecuta la siguiente celda para comenzar a entrenar el modelo.

In [ ]:
trained_model, losses = training_loop(embedding_model, training_pairs)

#### Visualizar la pérdida del entrenamiento (Training Loss)

* Ejecuta la siguiente celda para graficar los valores de pérdida de cada época.

In [ ]:
helper_utils.plot_loss(losses)

### Explorando la similitud semántica

Ahora que el modelo está entrenado, puedes verificar si ha aprendido relaciones significativas. Harás esto usando la misma métrica de **similitud de coseno** que exploraste anteriormente con los embeddings de GloVe.

El objetivo es observar puntuaciones de similitud altas para palabras dentro de la misma categoría (por ejemplo, 'car' y 'bike') y puntuaciones bajas para palabras en categorías diferentes (por ejemplo, 'car' y 'cat'). Esto confirmará que el entrenamiento fue exitoso.

* Define la función `cosine_similarity_words`.
    * La función tomará dos palabras como entrada y devolverá su puntuación de similitud basada en los embeddings aprendidos.

In [ ]:
def cosine_similarity_words(word1, word2, word_to_idx, embeddings_matrix):
    """
    Calcula la similitud de coseno entre dos palabras.

    Argumentos:
        word1 (str): La primera palabra a comparar.
        word2 (str): La segunda palabra a comparar.
        word_to_idx (dict): Un mapeo de palabras a sus índices.
        embeddings_matrix (np.ndarray): La matriz que contiene todos los vectores de palabras.
    """
    # Obtener los índices de las palabras
    idx1 = word_to_idx[word1]
    idx2 = word_to_idx[word2]

    # Extraer los vectores correspondientes de la matriz de embeddings
    emb1 = embeddings_matrix[idx1]
    emb2 = embeddings_matrix[idx2]

    # Calcular la similitud de coseno entre ambos vectores
    similarity = cosine_similarity([emb1], [emb2])[0][0]

    return similarity

* Extrae todos los vectores de palabras finales del `trained_model`. La forma más eficiente es tomar la matriz completa de embeddings a la vez.

    * `trained_model.embedding.weight`: Accede directamente a los vectores de palabras finales y aprendidos desde la matriz de pesos de la capa de embedding.
    * `.detach().numpy()`: Convierte la matriz de vectores de un tensor de PyTorch a un arreglo de NumPy, facilitando su manipulación.


In [ ]:
# Set the model to evaluation mode.
trained_model.eval()

# Extract the embedding matrix.
all_embeddings = trained_model.embedding.weight.detach().numpy()

* Define los pares de prueba.

In [ ]:
similarity_tests = [
    ("car","car"),
    ("car","bike"),
    ("car","plane"),

    ("car","cat"),
    ("car","dog"),
    ("car","bird"),

    ("car","orange"),
    ("car","apple"),
    ("car","grape"),

]

* Finalmente, recorre los pares de prueba, calcula la puntuación de similitud para cada uno e imprime los resultados.

In [ ]:
print("Similitud Semántica (Similitud de Coseno):")
print("="*40)

# Recorrer cada par de palabras en la lista de prueba.
for word1, word2 in similarity_tests:
    # Calcular la puntuación de similitud para el par actual
    similarity = cosine_similarity_words(word1, word2, word_to_idx, all_embeddings)
    
    # Imprimir el par de palabras y su similitud calculada
    print(f"{word1} <-> {word2}:\t {similarity:.4f}")

* Para obtener una visión más completa en lugar de solo probar unos pocos pares, calcula la puntuación de similitud entre cada combinación posible de palabras en tu vocabulario.

In [ ]:
# Inicializar una matriz cuadrada vacía con ceros para contener las puntuaciones de similitud.
similarity_matrix = np.zeros((vocab_size, vocab_size))

# Iterar a través de cada fila de la matriz (que representa la primera palabra).
for i in range(vocab_size):
    # Iterar a través de cada columna (que representa la segunda palabra).
    for j in range(vocab_size):
        # Para cualquier palabra comparada consigo misma, la similitud es un 1.0 perfecto.
        if i == j:
            similarity_matrix[i, j] = 1.0
        # Para pares de palabras diferentes:
        else:
            # Obtener la representación en cadena de cada palabra a partir de sus índices.
            word1 = idx_to_word[i]
            word2 = idx_to_word[j]
            # Calcular la similitud y colocarla en la celda correcta de la matriz.
            similarity_matrix[i, j] = cosine_similarity_words(word1, word2, word_to_idx, all_embeddings)

* Ejecuta la siguiente celda para graficar la `similarity_matrix`.

In [ ]:
helper_utils.plot_similarity_matrix(similarity_matrix, vocabulary)

La matriz resultante muestra claramente que las palabras de la categoría `Vehicle` tienen una alta similitud entre ellas, pero una baja similitud con las palabras de las categorías `Pet` y `Fruit`, y viceversa.

## Visualización de tus word embeddings

Ahora que tu modelo básico está entrenado, es hora de ver qué tan bien capturó el significado semántico de tu pequeño vocabulario. Luego, podrás comparar tus resultados con los grupos (clusters) que viste anteriormente con el modelo preentrenado GloVe.

* Primero, utiliza PCA nuevamente para reducir tus vectores de 3 dimensiones a 2 dimensiones.

In [ ]:
# Reduce dimensionality
reducer = PCA(n_components=2)
coords = reducer.fit_transform(all_embeddings)

* Ejecuta la siguiente celda para graficar tus word embeddings entrenados.

In [ ]:
helper_utils.plot_embeddings(coords=coords, 
                             labels=vocabulary,
                             label_dict=vocab_categories,
                             title='Your Trained Word Embeddings'
                            )

El gráfico confirma que el entrenamiento fue un éxito. El resultado es conceptualmente similar al gráfico de GloVe que viste antes. Las palabras se han separado una vez más en sus tres categorías semánticas distintas.

Esto demuestra de manera contundente que incluso el modelo simple que construiste desde cero aprendió eficazmente las relaciones entre las palabras, probando el principio fundamental detrás de los *word embeddings*.

## Más allá de los embeddings estáticos: La importancia del contexto

Aunque entrenar tus propios embeddings es una forma fantástica de entender cómo funcionan, en muchas aplicaciones del mundo real no es necesario empezar desde cero. Como viste en la primera mitad de este cuaderno, los modelos preentrenados ofrecen un punto de partida potente y eficiente.

Los **modelos estáticos** como **GloVe** son herramientas excelentes y computacionalmente ligeras, perfectas para diversas tareas donde el significado general de las palabras es suficiente. Son una gran elección para aplicaciones como:

* Análisis de sentimiento general.
* Clasificación de documentos o modelado de temas.
* Escenarios donde la velocidad y un menor uso de memoria son importantes.

### La limitación: El problema del "bate" (The "Bat" Problem)

Sin embargo, el enfoque de "una palabra, un vector" de los modelos estáticos tiene una limitación clave cuando se trata de palabras que tienen múltiples significados (un concepto conocido como **polisemia**).

Por ejemplo, considera la palabra "bat" en estas dos frases:

> 1. "A **bat** flew out of the cave." (un animal: murciélago)
>
> 2. "He swung the baseball **bat**." (equipo deportivo: bate)

Un modelo estático como GloVe producirá exactamente el mismo vector para "bat" en ambos casos. Este vector es un promedio de todos los contextos que el modelo vio durante su entrenamiento, por lo que no puede distinguir entre el animal y el equipo deportivo.

Veamos esto en acción.

* Primero, define las dos oraciones que usarás para la comparación.

In [ ]:
# Las oraciones para la comparación
sentence1 = "A bat flew out of the cave."
sentence2 = "He swung the baseball bat."

* A continuación, recupera el vector preentrenado de GloVe específico para la palabra `"bat"`.

In [ ]:
# Obtener los vectores específicos para "bat" de cada oración
bat_from_sentence1 = glove_embeddings["bat"]
bat_from_sentence2 = glove_embeddings["bat"]

* Imprime los vectores para ambas oraciones.

In [ ]:
# --- Imprimir vectores para la primera oración ---
print("--- Oración 1 (primeros 5 valores) ---")
for word in sentence1.split():
    # Limpiar la palabra para eliminar signos de puntuación comunes
    clean_word = word.strip('.,?!').lower()
    
    # Verificar si la palabra limpia existe en el vocabulario de GloVe
    if clean_word in glove_embeddings:
        vector = glove_embeddings[clean_word]
        print(f"{clean_word:<12} {vector[:5]}")
    else:
        print(f"{clean_word:<12} {'(no está en el vocabulario)'}")

In [ ]:
# --- Imprimir vectores para la segunda oración ---
print("--- Oración 2 (primeros 5 valores) ---")
for word in sentence2.split():
    # Limpiar la palabra para eliminar signos de puntuación comunes
    clean_word = word.strip('.,?!').lower()
    
    # Verificar si la palabra limpia existe en el vocabulario de GloVe
    if clean_word in glove_embeddings:
        vector = glove_embeddings[clean_word]
        print(f"{clean_word:<12} {vector[:5]}")
    else:
        print(f"{clean_word:<12} {'(no está en el vocabulario)'}")

* Como confirmación final, compara los dos vectores recuperados para la palabra "bat" para demostrar que son idénticos.

In [ ]:
# Verificar si los dos vectores para 'bat' son idénticos
are_identical = np.array_equal(bat_from_sentence1, bat_from_sentence2)
print(f"¿Son idénticos los vectores para 'bat' de cada oración? {are_identical}")

### La solución: Embeddings contextuales con BERT

Aquí es donde entran en juego modelos avanzados como [BERT](https://huggingface.co/docs/transformers/en/model_doc/bert) (Bidirectional Encoder Representations from Transformers). A diferencia de los modelos estáticos, BERT es **contextual**.

Esto significa que genera un vector único y dinámico para una palabra cada vez que aparece, basándose en la oración específica en la que se encuentra.

Veamos esto en acción.

* Asegúrate de que el modelo se haya descargado localmente.

In [ ]:
helper_utils.download_bert()

* Carga el tokenizador y el modelo en sus respectivas variables.

In [ ]:
# Definir la ruta local donde está guardado el modelo BERT.
bert_path = './bert_model'

# Cargar el tokenizador y el modelo desde la ruta especificada.
tokenizer, model_bert = helper_utils.load_bert(bert_path)

Imprime el vector contextual para cada token en las oraciones.

* Primero, procesa la primera oración. Utiliza el tokenizador de BERT para preparar la entrada y luego alimentarla al modelo.
* A continuación, extrae el `last_hidden_state`, que contiene el embedding final y sensible al contexto para cada token de la oración.

* Finalmente, recorre cada token y su vector correspondiente para imprimirlos lado a lado.

**Nota**: El punto (`.`) aparece como un **token** separado porque el tokenizador avanzado de BERT reconoce que la puntuación tiene un significado gramatical; a diferencia del método simple `.split()` utilizado en el ejemplo de GloVe (que requería eliminar el punto manualmente), el tokenizador de BERT lo preserva intencionalmente como una parte significativa de la estructura de la oración.

In [ ]:
# --- Procesar e imprimir vectores para la Oración 1 ---
print("--- Oración 1 (primeros 5 valores) ---")
# Tokenizar la oración y obtener la salida del modelo
inputs1 = tokenizer(sentence1, return_tensors='pt')
with torch.no_grad():
    outputs1 = model_bert(**inputs1)
last_hidden_state1 = outputs1.last_hidden_state[0] # Embeddings para todos los tokens

# Obtener los tokens reales a partir de sus IDs
tokens1 = tokenizer.convert_ids_to_tokens(inputs1['input_ids'][0])

# Recorrer cada token y su vector correspondiente
for token, vector in zip(tokens1, last_hidden_state1):
    # Imprimir el token y las primeras 5 dimensiones de su vector contextual
    print(f"{token:<12} {vector.numpy()[:5]}")

* Ahora realiza lo mismo con la segunda oración.

In [ ]:
# --- Procesar e imprimir vectores para la Oración 2 ---
print("--- Oración 2 (primeros 5 valores) ---")
# Tokenizar la oración y obtener la salida del modelo
inputs2 = tokenizer(sentence2, return_tensors='pt')
with torch.no_grad():
    outputs2 = model_bert(**inputs2)
last_hidden_state2 = outputs2.last_hidden_state[0] # Embeddings para todos los tokens

# Obtener los tokens reales a partir de sus IDs
tokens2 = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])

# Recorrer cada token y su vector correspondiente
for token, vector in zip(tokens2, last_hidden_state2):
    # Imprimir el token y las primeras 5 dimensiones de su vector contextual
    print(f"{token:<12} {vector.numpy()[:5]}")

* También puedes realizar la misma verificación de identidad en los vectores de BERT.
    * El resultado será `False`, confirmando que BERT produjo dos vectores únicos para la palabra `"bat"`.

In [ ]:
# Extraer el vector para "bat" de la primera oración (en el índice de token 2)
bat_animal_vector = last_hidden_state1[2].numpy()
# Extraer el vector para "bat" de la segunda oración (en el índice de token 5)
bat_sport_vector = last_hidden_state2[5].numpy()
# Verificar si los dos vectores contextuales para "bat" son idénticos
are_identical = np.array_equal(bat_animal_vector, bat_sport_vector)
print(f"¿Son idénticos los vectores contextuales de BERT para 'bat'? {are_identical}")

## La conclusión: ¿Qué modelo deberías usar?

Después de ver la diferencia, la pregunta natural es qué tipo de embedding utilizar. La respuesta depende enteramente de tu tarea específica y de tus recursos.

**Usa Embeddings Estáticos (como GloVe) cuando:**

* Estés realizando una tarea directa, como la clasificación de documentos, donde el contexto matizado es menos crítico.
* Necesites una solución rápida, computacionalmente ligera y con un menor uso de memoria.

**Usa Embeddings Contextuales (como BERT) cuando:**

* Tu tarea requiera una comprensión profunda del lenguaje y la ambigüedad (por ejemplo, sistemas de preguntas y respuestas, resumen de textos o chatbots avanzados).
* El manejo de palabras con múltiples significados sea importante para el éxito de tu aplicación.
* Tengas los recursos computacionales para ejecutar modelos más grandes y complejos.

## Conclusión
En este laboratorio, has recorrido el proceso completo de trabajar con word embeddings, pasando de números abstractos a representaciones vectoriales ricas y significativas. Comenzaste cargando un potente modelo estático preentrenado, GloVe, y viste de primera mano cómo sus vectores capturan relaciones semánticas complejas, permitiéndote resolver analogías con aritmética simple. La visualización de estos embeddings con PCA dejó claro cómo las palabras con significados similares se agrupan en el espacio vectorial.

Luego, construiste tu propio modelo de embeddings desde cero, obteniendo una comprensión fundamental de cómo se aprenden estas representaciones durante el entrenamiento. Esto resaltó una limitación fundamental de los modelos estáticos: su incapacidad para manejar palabras con múltiples significados. Para solucionar esto, exploraste BERT, un modelo contextual que genera embeddings únicos basados en el texto que rodea a una palabra, proporcionando una comprensión mucho más matizada del lenguaje.

Ahora posees las habilidades prácticas para cargar, entrenar, visualizar y diferenciar entre embeddings estáticos y contextuales. Este conocimiento es una base esencial para construir redes neuronales más avanzadas que puedan realizar tareas sofisticadas de PLN, como la clasificación de texto y el análisis de sentimiento.